# 🚁 Tello EDU : Mode Réel (Caméra Drone)

Ce notebook connecte le script `vision.py` et `tello_controller.py` au **vrai drone Tello** au lieu de la webcam du PC.

**Prérequis :**
1. Allumez le Tello.
2. Connectez le WiFi de votre PC au réseau du Tello (ex: `TELLO-XXXXXX`).
3. Vérifiez que `djitellopy` est installé.

In [ ]:
%matplotlib inline
import sys
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

# Import des modules du projet
from tello_controller import TelloController
from vision import VideoStream, ObstacleDetector

print("Modules chargés.")

### 1. Connexion au Drone
Nous initialisons le contrôleur en mode `simulation_mode=False`. Cela va tenter d'établir une connexion UDP avec le drone.

In [ ]:
# Initialisation du contrôleur en mode RÉEL
controller = TelloController(simulation_mode=False)

print("Tentative de connexion au drone...")
if controller.connect():
    print("✅ Connecté au Tello !")
    telemetry = controller.get_telemetry()
    print(f"Batterie : {telemetry.get('battery')}% | Température : {telemetry.get('temperature')}°C")
else:
    print("❌ Échec de connexion. Vérifiez le WiFi.")

### 2. Démarrage du Flux Vidéo (Contexte RGB)
Ici, nous passons l'objet `controller.drone` (l'instance `djitellopy`) à la classe `VideoStream`.

**Important :** OpenCV et le Tello fournissent généralement du **BGR**. Pour l'affichage dans ce notebook, nous convertissons en **RGB**.

In [ ]:
# On passe l'objet drone réel au VideoStream
video_stream = VideoStream(drone=controller.drone, simulation_mode=False)
detector = ObstacleDetector()

# Démarrage du flux (commande 'streamon' envoyée au drone)
video_stream.start()
print("Flux vidéo démarré. Patientez quelques secondes pour la stabilisation...")
time.sleep(3)

### 3. Affichage en temps réel
Cette boucle récupère les frames, détecte les obstacles, convertit les couleurs (BGR -> RGB) et affiche le résultat.

In [ ]:
try:
    # Boucle d'affichage (arrêter avec le bouton Stop du notebook)
    for i in range(200):  # Limité à 200 frames pour l'exemple
        
        # 1. Récupération de la frame brute (Format BGR du Tello/OpenCV)
        frame_bgr = video_stream.get_frame()
        
        if frame_bgr is not None:
            # 2. Détection d'obstacles (se fait sur du BGR ou Gris en interne)
            obstacles = detector.detect(frame_bgr)
            
            # 3. Dessin des annotations (sur l'image BGR)
            frame_annotated_bgr = detector.draw_detections(frame_bgr, obstacles)
            
            # 4. Conversion BGR -> RGB pour l'affichage Notebook
            # C'est ici qu'on respecte la contrainte "RGB" pour l'utilisateur
            frame_rgb = cv2.cvtColor(frame_annotated_bgr, cv2.COLOR_BGR2RGB)
            
            # 5. Affichage Matplotlib dynamique
            clear_output(wait=True)
            plt.figure(figsize=(10, 6))
            plt.imshow(frame_rgb)
            plt.axis('off')
            plt.title(f"Flux Tello (RGB) - Obstacles: {len(obstacles)}")
            plt.show()
        
        # Petite pause pour ne pas surcharger le notebook
        time.sleep(0.05)
        
except KeyboardInterrupt:
    print("Arrêt par l'utilisateur")
except Exception as e:
    print(f"Erreur : {e}")
finally:
    # Nettoyage propre
    print("Arrêt du flux et déconnexion...")
    video_stream.stop()
    controller.disconnect()